In [ ]:

import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from scipy.spatial.distance import euclidean
from sklearn.metrics.pairwise import linear_kernel
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
#Loading data set
df=pd.read_csv(r'./data/laptop.csv', encoding="ISO-8859-1")
df

In [4]:
df.head()

,Unnamed: 0,Brand,Name,Price,Processor_Name,Processor_Brand,RAM_Expandable,RAM,RAM_TYPE,Ghz,Display_type,Display,GPU,GPU_Brand,SSD,HDD,Adapter,Battery_Life
0,0,HP,HP Chromebook 11A-NA0002MU (2E4N0PA) Laptop (1...,22990,MediaTek Octa-core,MediaTek,Not Expandable,4 GB,DDR4 RAM,2.0 Ghz Processor,LED,11.6,Integrated Graphics,MediaTek,64 GB SSD Storage,No HDD,45,Upto 12 Hrs Battery Life
1,1,Lenovo,Lenovo Ideapad Slim 3 (82KU017KIN) Laptop (15....,36289,AMD Hexa-Core Ryzen 5,AMD,12 GB Expandable,8 GB,DDR4 RAM,4.0 Ghz Processor,LCD,15.6,Radeon,AMD,512 GB SSD Storage,No HDD,65,Upto 11 Hrs Battery Life
2,3,Dell,Dell G15-5520 (D560822WIN9B) Laptop (15.6 Inch...,78500,Intel Core i5 (12th Gen),Intel,32 GB Expandable,16 GB,DDR5 RAM,3.3 Ghz Processor,LCD,15.6,"GeForce RTX 3050 GPU, 4 GB",NVIDIA,512 GB SSD Storage,No HDD,56,Upto 10 Hrs Battery Life
3,4,HP,HP 15s-fy5007TU (91R03PA) Laptop (15.6 Inch | ...,55490,Intel Core i5 (12th Gen),Intel,8 GB Expandable,8 GB,DDR4 RAM,4.2 Ghz Processor,LCD,15.6,Iris Xe,Intel,512 GB SSD Storage,No HDD,no,Upto 7.30 Hrs Battery Life
4,6,Infinix,Infinix Inbook Y2 Plus XL29 Laptop (15.6 Inch ...,21990,Intel Core i3 (11th Gen),Intel,Not Expandable,8 GB LP,LPDDR4X RAM,1.7 Ghz Processor,LCD,15.6,UHD,Intel,512 GB SSD Storage,No HDD,45,Upto 8 Hrs Battery Life


In [5]:
# Display the data types and dataset information
print("\nDataset Information:")
df.info()


Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 3976 entries, 0 to 3975
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Unnamed: 0       3976 non-null   int64
 1   Brand            3976 non-null   str  
 2   Name             3976 non-null   str  
 3   Price            3976 non-null   int64
 4   Processor_Name   3976 non-null   str  
 5   Processor_Brand  3976 non-null   str  
 6   RAM_Expandable   3976 non-null   str  
 7   RAM              3976 non-null   str  
 8   RAM_TYPE         3976 non-null   str  
 9   Ghz              3976 non-null   str  
 10  Display_type     3976 non-null   str  
 11  Display          3976 non-null   str  
 12  GPU              3968 non-null   str  
 13  GPU_Brand        3972 non-null   str  
 14  SSD              3976 non-null   str  
 15  HDD              3976 non-null   str  
 16  Adapter          3976 non-null   str  
 17  Battery_Life     3558 non-null   str  
dt

In [6]:
df.describe()

,Unnamed: 0,Price
count,3976.000000,3976.000000
mean,2181.495724,72432.528672
std,1297.029657,52207.650948
min,0.000000,7990.000000
25%,1058.750000,39873.250000
50%,2098.500000,58990.000000
75%,3342.250000,84990.000000
max,4408.000000,503890.000000


In [7]:
df.isnull().sum()

Unnamed: 0           0
Brand                0
Name                 0
Price                0
Processor_Name       0
Processor_Brand      0
RAM_Expandable       0
RAM                  0
RAM_TYPE             0
Ghz                  0
Display_type         0
Display              0
GPU                  8
GPU_Brand            4
SSD                  0
HDD                  0
Adapter              0
Battery_Life       418
dtype: int64

In [8]:
# Check and print the number of duplicate rows
duplicate_count_before = df.duplicated().sum()
print("Total number of duplicate rows:", duplicate_count_before)

Total number of duplicate rows: 0


In [9]:
# Check the number of unique values
df.nunique()

Unnamed: 0         3976
Brand                31
Name               3941
Price              1799
Processor_Name      125
Processor_Brand      19
RAM_Expandable       10
RAM                  20
RAM_TYPE             19
Ghz                  31
Display_type          2
Display              34
GPU                 300
GPU_Brand            11
SSD                  19
HDD                   8
Adapter              68
Battery_Life        191
dtype: int64

In [12]:
# Check if 'laptop_ID' is still in the dataset
print("laptop_ID" in df.columns)
print("Remaining columns:", df.columns)

False
Remaining columns: Index(['Unnamed: 0', 'Brand', 'Name', 'Price', 'Processor_Name',
       'Processor_Brand', 'RAM_Expandable', 'RAM', 'RAM_TYPE', 'Ghz',
       'Display_type', 'Display', 'GPU', 'GPU_Brand', 'SSD', 'HDD', 'Adapter',
       'Battery_Life'],
      dtype='str')


In [15]:
#  5 rows before transformation
print("Before Removing 'GB' from Ram:")
print(df[["RAM_TYPE"]].head())

df["RAM_TYPE"] = df["RAM_TYPE"].astype(str).str.replace("GB", "").astype(int)

#  first 5 rows after transformation
print("\nAfter Removing 'GB' from Ram:")
print(df[["RAM_TYPE"]].head())

Before Removing 'GB' from Ram:
       RAM_TYPE
0      DDR4 RAM
1     DDR4 RAM 
2     DDR5 RAM 
3     DDR4 RAM 
4   LPDDR4X RAM


ValueError: invalid literal for int() with base 10: ' DDR4 RAM'